# Cornell potential fit plot

This notebook reads the precomputed Cornell-fit output and plots the stored Cornell fit for `r_min=4` together with the finalized static-potential data. The figure is saved as a PDF without a title.


In [2]:
from pathlib import Path
import json
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from calculator import cornell_potential_ansatz
from finalized_analysis_helpers import _ensure_plotly_browser_path

POTENTIAL_DIR = Path("../data/gradient_flow_wtemp_analysis/beta_2p5__L0_32__epsilon1_0__dt_0p01__72d8fe58/potential_t_over_a2_0p5")
CORNELL_DIR = POTENTIAL_DIR / "cornell_fit"
R_MIN = 4
PDF_PATH = CORNELL_DIR / f"cornell_fit_rmin{R_MIN}.pdf"

# Set these to tuples like (xmin, xmax) / (ymin, ymax), or leave as None for automatic limits.
XLIM = None
YLIM = None

# Plot styling, matching wilson_potential_fit_plotting.ipynb PDF exports.
FIGURE_WIDTH = 950
FIGURE_HEIGHT = 600
POINT_COLOR = "#1f77b4"
FIT_COLOR = "#d62728"
EXCLUDED_COLOR = "#7f7f7f"
R0_COLOR = "#2ca02c"
R0_FORCE_TARGET = 1.65
CAPSIZE = 3
MARKER_SIZE = 7
PDF_MARGIN = {"l": 48, "r": 12, "t": 12, "b": 46}

_ensure_plotly_browser_path()
pio.templates.default = "plotly_white"

results_path = CORNELL_DIR / "cornell_fit_results.json"
with results_path.open("r", encoding="utf-8") as handle:
    cornell_results = json.load(handle)

fit_rows = {int(row["r_min"]): row for row in cornell_results.get("fits", [])}
fit_info = fit_rows.get(int(R_MIN))
if fit_info is None:
    available = ", ".join(str(value) for value in sorted(fit_rows))
    raise ValueError(f"No Cornell fit for r_min={R_MIN}. Available r_min values: {available}")

points = sorted(cornell_results.get("potential_points", []), key=lambda row: int(row["R"]))
if not points:
    raise ValueError(f"No potential points found in {results_path}")

R = np.asarray([int(row["R"]) for row in points], dtype=float)
V = np.asarray([float(row["V"]) for row in points], dtype=float)
V_err = np.asarray(
    [float(row["err"]) if row.get("err") is not None else np.nan for row in points],
    dtype=float,
)
used_mask = R >= float(R_MIN)
if fit_info.get("r_max") is not None:
    used_mask &= R <= float(fit_info["r_max"])

params = fit_info["cornell_params"]
A = float(params["A"])
sigma = float(params["sigma"])
B = float(params["B"])
r0 = float(fit_info["r0"])
r0_err = fit_info.get("r0_err")
chi2_dof = fit_info.get("chi2_dof")

def _fmt_err(value):
    return "n/a" if value is None or not np.isfinite(float(value)) else f"{float(value):.3g}"

print("r_min:", R_MIN)
print("A:", f"{A:.8g}")
print("sigma:", f"{sigma:.8g}")
print("B:", f"{B:.8g}")
print("r0/a:", f"{r0:.8g}", "+/-", _fmt_err(r0_err))
print("chi^2/dof:", "n/a" if chi2_dof is None else f"{float(chi2_dof):.6g}")

r_line = np.linspace(float(np.min(R)), float(np.max(R)), 500)
v_fit = cornell_potential_ansatz(r_line, A, sigma, B)
v_r0 = float(cornell_potential_ansatz(np.asarray([r0]), A, sigma, B)[0])

def _scatter_trace(x, y, yerr, *, name, color, symbol="circle"):
    error_array = np.where(np.isfinite(yerr), yerr, 0.0)
    return go.Scatter(
        x=x,
        y=y,
        mode="markers",
        name=name,
        marker={"size": MARKER_SIZE, "color": color, "symbol": symbol},
        error_y={"type": "data", "array": error_array, "visible": True, "thickness": 1, "width": CAPSIZE},
        customdata=np.column_stack([error_array]),
        hovertemplate=(
            "R=%{x:g}<br>"
            "V=%{y:.8g}<br>"
            "bootstrap error=%{customdata[0]:.3g}"
            "<extra></extra>"
        ),
    )


def make_cornell_fit_pdf_figure(*, xlim=None, ylim=None):
    fig = go.Figure()
    excluded = ~used_mask
    if np.any(excluded):
        fig.add_trace(
            _scatter_trace(
                R[excluded],
                V[excluded],
                V_err[excluded],
                name="outside fit range",
                color=EXCLUDED_COLOR,
                symbol="circle-open",
            )
        )

    fig.add_trace(
        _scatter_trace(
            R[used_mask],
            V[used_mask],
            V_err[used_mask],
            name="data used in fit",
            color=POINT_COLOR,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=r_line,
            y=v_fit,
            mode="lines",
            name="fit",
            line={"color": FIT_COLOR, "width": 2},
            hovertemplate="R=%{x:.3g}<br>fit=%{y:.8g}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[r0],
            y=[v_r0],
            mode="markers",
            name="r0 condition",
            marker={
                "size": 12,
                "color": R0_COLOR,
                "symbol": "diamond",
                "line": {"color": "white", "width": 1},
            },
            hovertemplate=(
                "r0=%{x:.5g}<br>"
                "V(r0)=%{y:.8g}<br>"
                f"r0^2 F(r0)={R0_FORCE_TARGET:g}"
                "<extra></extra>"
            ),
        )
    )


    fig.update_layout(
        title=None,
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        xaxis_title="R",
        yaxis_title="V(R)",
        margin=PDF_MARGIN,
        showlegend=True,
        legend={"x": 0.02, "xanchor": "left", "y": 0.98, "yanchor": "top"},
    )
    fig.update_xaxes(dtick=2)
    if xlim is not None:
        fig.update_xaxes(range=list(xlim))
    if ylim is not None:
        fig.update_yaxes(range=list(ylim))
    return fig

fig = make_cornell_fit_pdf_figure(xlim=XLIM, ylim=YLIM)
PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_image(str(PDF_PATH))
print(f"Saved PDF: {PDF_PATH}")
fig


r_min: 4
A: 0.12377229
sigma: 0.035366476
B: 0.24413914
r0/a: 6.3048576 +/- 0.0089
chi^2/dof: 1.0025
Saved PDF: ../data/gradient_flow_wtemp_analysis/beta_2p5__L0_32__epsilon1_0__dt_0p01__72d8fe58/potential_t_over_a2_0p5/cornell_fit/cornell_fit_rmin4.pdf
